# Breast Cancer Molecular Subtype Classification from Whole-Slide Images
## Approach 1 (baseline) — KimiaNet features from patch mosaics

**KimiaNet** features (1024-D) extracted from mosaics of 16 patches, built by *k*-medoids clustering of the patch colour descriptors under four distances (`cosine`, `euclidean`, `bhattacharyya`, `jensenshannon`).

## Overview

Each slide is summarised by a single 4000 × 4000 px mosaic: its patches are grouped into 16 histological prototypes by *k*-medoids on the RGB histogram, and the mosaic is then represented by the 1024 KimiaNet features.

For every one of the four distances used to build the mosaics, a panel of five classifiers (KNN, RF, LR, SVM, XGBoost) is trained and evaluated, so that the choice of distance is treated as a design variable rather than fixed in advance. An MLP is added as a sixth classifier.

The train/test split is the **same patient-level partition used in the transcriptomic phase**. Keeping it identical is what makes the two modalities paired, which is required for the later multi-omic integration.

- **Inputs:** one `KimiaNet_features.csv` per distance (TCGA cohort and the INC cohort) and `split_transcriptomica_604.csv`.
- **Outputs:** trained models, summary tables, confusion matrices and the external-validation report in `OUT_DIR`.

## Contents

1. Configuration and imports
2. Classical classifiers per distance
3. Evaluation by repeated cross-validation
4. External validation — INC cohort
5. MLP on the KimiaNet features
6. Notes and limitations

## 1. Configuration and imports

### 1.1 Imports

Everything used in the notebook is imported here: scikit-learn for the classical models and the metrics, XGBoost for the boosted trees, PyTorch for the MLP of Section 5, and `joblib` to persist the tuned models.

In [ ]:
# ── Imports (consolidated) ───────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

from scipy.stats import randint, uniform, wilcoxon
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (StratifiedKFold, GridSearchCV, RandomizedSearchCV,
                                     RepeatedStratifiedKFold, cross_validate)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, silhouette_score, adjusted_rand_score)
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
import umap

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

### 1.2 Paths and constants

`RUTA_FEATURES` is a template: the `{metrica}` placeholder is filled with each of the four distances in turn. The class encoding and the cross-validation schemes are the same as in the transcriptomic phase, so the two modalities are compared under one protocol.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
METRICAS      = ["cosine", "euclidean", "bhattacharyya", "jensenshannon"]
RUTA_FEATURES = "mosaics/{metrica}/KimiaNet_features_completo.csv"   # adjust
RUTA_SPLIT    = "split_transcriptomica_604.csv"                      # partition from the transcriptomic notebook
OUT_DIR       = Path("modelos_wsi")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS  = [str(i) for i in range(1024)]
MAP           = {"LumA": 0, "LumB": 1, "Basal": 2}
CLASES        = ["LumA", "LumB", "Basal"]          # order 0, 1, 2 (consistent throughout)
cv            = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_report     = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

df_split = pd.read_csv(RUTA_SPLIT)
print("Split:", df_split["split"].value_counts().to_dict())

### 1.3 Loading one distance

The features of a given distance are merged with the partition on the patient identifier. The merge is an inner join, so only the patients present in both tables survive: a slide without transcriptomic data, or a patient without a mosaic, is dropped silently. It is worth printing the resulting sizes to confirm the expected 386 / 166.

`np.nan_to_num` replaces missing or infinite values with zero, since the distance-based models cannot handle NaN.

In [ ]:
# ── Load and label one distance ──────────────────────────────────────────────
def cargar_metrica(metrica):
    """Return (X_train, X_test, y_train, y_test, test_ids) for one mosaic distance."""
    df = pd.read_csv(RUTA_FEATURES.format(metrica=metrica))
    df = df.merge(df_split[["patient_id", "Subtype", "split"]],
                  left_on="ID", right_on="patient_id", how="inner")

    tr, te = df[df.split == "train"], df[df.split == "test"]
    Xtr = np.nan_to_num(tr[FEATURE_COLS].values.astype(float))   # distances: no NaN/Inf allowed
    Xte = np.nan_to_num(te[FEATURE_COLS].values.astype(float))
    ytr = tr["Subtype"].map(MAP).values
    yte = te["Subtype"].map(MAP).values
    return Xtr, Xte, ytr, yte, te["ID"].values

## 2. Classical classifiers

### 2.1 Search spaces

The five classifiers are declared in a single dictionary, each with its search strategy (`grid` or `random`) and its parameter space. The scaler sits inside the pipeline for the scale-sensitive models (KNN, LR, SVM) and is omitted for the tree-based ones (RF, XGBoost), which do not need it.

Placing the scaler inside the pipeline matters: it is re-fitted within each cross-validation fold, so the validation fold never contributes to the mean and variance used to standardise the training data.

In [ ]:
# ── The five classifiers ─────────────────────────────────────────────────────
# Scaler INSIDE the pipeline for the scale-sensitive models (KNN, LR, SVM);
# RF and XGBoost do not need it (trees). Same architecture as the transcriptomic phase.
def espacios_busqueda():
    return {
        "KNN": ("grid",
            Pipeline([("sc", StandardScaler()), ("clf", KNeighborsClassifier())]),
            {"clf__n_neighbors": list(range(1, 31)), "clf__weights": ["uniform", "distance"],
             "clf__metric": ["minkowski", "cosine"], "clf__p": [1, 2, 3]}),
        "Random Forest": ("random",
            RandomForestClassifier(random_state=42, n_jobs=-1),
            {"n_estimators": [100, 200, 300, 500], "max_depth": [None, 30, 40, 50],
             "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 5],
             "max_features": [0.02, 0.05, 0.1, 0.15, 0.18], "criterion": ["gini", "entropy"]}),
        "Logistic Regression": ("grid",
            Pipeline([("sc", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=5000, random_state=42))]),
            {"clf__C": [0.01, 0.1, 1, 10], "clf__penalty": ["l2"], "clf__solver": ["lbfgs"]}),
        "SVM": ("grid",
            Pipeline([("sc", StandardScaler()), ("clf", SVC(random_state=42))]),
            {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__kernel": ["rbf", "poly"],
             "clf__gamma": ["scale", 1, 0.1, 0.01, 0.001, 0.0001], "clf__degree": [2, 3, 4],
             "clf__coef0": [0.0, 0.1, 1.0]}),
        "XGBoost": ("random",
            XGBClassifier(objective="multi:softprob", num_class=3, tree_method="hist",
                          eval_metric="mlogloss", n_jobs=-1, random_state=42),
            {"n_estimators": randint(200, 600), "max_depth": randint(3, 7),
             "learning_rate": uniform(0.03, 0.15), "subsample": uniform(0.6, 0.3),
             "colsample_bytree": uniform(0.4, 0.5), "min_child_weight": randint(1, 6)}),
    }


def entrenar_evaluar(Xtr, Xte, ytr, yte, n_iter=50):
    """Tune every classifier on the training set and score it on the test set.

    Returns the dictionary of tuned models and the summary table sorted by F1-macro.
    """
    modelos, filas = {}, []
    for nombre, (tipo, est, space) in espacios_busqueda().items():
        if tipo == "grid":
            search = GridSearchCV(est, space, cv=cv, scoring="f1_macro", n_jobs=-1)
        else:
            search = RandomizedSearchCV(est, space, n_iter=n_iter, cv=cv,
                                        scoring="f1_macro", random_state=42, n_jobs=-1)
        search.fit(Xtr, ytr)
        best = search.best_estimator_
        yp   = best.predict(Xte)
        modelos[nombre] = best
        filas.append({
            "Model":     nombre,
            "Accuracy":  round(accuracy_score(yte, yp), 3),
            "Precision": round(precision_score(yte, yp, average="macro", zero_division=0), 3),
            "Recall":    round(recall_score(yte, yp, average="macro", zero_division=0), 3),
            "F1_macro":  round(f1_score(yte, yp, average="macro"), 3),
        })
    tabla = pd.DataFrame(filas).set_index("Model").sort_values("F1_macro", ascending=False)
    return modelos, tabla

### 2.2 Training across the four distances

The whole panel is trained once per distance, giving 5 models × 4 distances = 20 tuned classifiers. Each is saved to `OUT_DIR` so the notebook does not need to be re-run to reuse them.

This loop is the expensive part of the notebook: the SVM grid alone holds 540 configurations, evaluated on five folds, for each of the four distances.

In [ ]:
# ── Loop over the four distances ─────────────────────────────────────────────
resultados, modelos_metrica = {}, {}
for metrica in METRICAS:
    Xtr, Xte, ytr, yte, ids_te = cargar_metrica(metrica)
    modelos, tabla = entrenar_evaluar(Xtr, Xte, ytr, yte)
    resultados[metrica]      = tabla
    modelos_metrica[metrica] = modelos

    print(f"\n===== {metrica.upper()}  (train={len(ytr)}, test={len(yte)}) =====")
    print(tabla.to_string())

    # Persist the models of this distance
    for nombre, mod in modelos.items():
        joblib.dump(mod, OUT_DIR / f"{nombre.split()[0].lower()}_{metrica}.pkl")

### 2.3 Comparison across distances

F1-macro of every model under every distance, in one table. Reading it by rows tells which classifier is the most robust; reading it by columns tells whether the distance used to build the mosaics has any systematic effect on the downstream classification.

The `Media` column averages over the four distances and is used to pick the model taken forward.

In [ ]:
# ── Global comparison table (F1-macro by model x distance) ───────────────────
comparativa = pd.concat({m: t["F1_macro"] for m, t in resultados.items()}, axis=1)
comparativa["Media"] = comparativa.mean(axis=1).round(3)
comparativa = comparativa.sort_values("Media", ascending=False)
print("F1-macro by model and mosaic distance:\n")
print(comparativa.to_string())

Confusion matrix of the selected pair (model, distance). Note that the distance is chosen by its single best value while the model is chosen by its mean across distances, so the two criteria differ.

In [ ]:
# ── Confusion matrix of the best (model, distance) pair ──────────────────────
mejor_metrica = comparativa.drop(columns="Media").max().idxmax()
mejor_modelo  = comparativa["Media"].idxmax()
Xtr, Xte, ytr, yte, _ = cargar_metrica(mejor_metrica)
yp = modelos_metrica[mejor_metrica][mejor_modelo].predict(Xte)

cm = confusion_matrix(yte, yp, labels=[0, 1, 2])
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix — {mejor_modelo} ({mejor_metrica})")
plt.tight_layout(); plt.show()
print(classification_report(yte, yp, labels=[0, 1, 2], target_names=CLASES))

## 3. Evaluation by repeated cross-validation

The test set holds 166 samples, so a difference of a few points between models may be noise. The tuned models are therefore re-evaluated on the whole cohort with 5-fold stratified cross-validation repeated three times (15 folds per model), which yields a mean and a dispersion instead of a single number.

Because every model sees the same folds, the per-fold scores are paired and the two best-ranked models can be compared with a Wilcoxon signed-rank test.

In [ ]:
scoring = {"F1_macro": "f1_macro", "Balanced_acc": "balanced_accuracy",
           "Precision": "precision_macro", "Recall": "recall_macro"}

In [ ]:
def reporte_cv(modelos, Xtr, Xte, ytr, yte):
    """Repeated stratified CV over the whole cohort (train + test combined).

    Returns the summary table (mean +/- SD) and the per-fold F1-macro scores,
    which are paired across models and feed the Wilcoxon test.
    """
    X = np.vstack([Xtr, Xte])
    y = np.concatenate([ytr, yte])
    filas, f1_folds = [], {}
    for nombre, mod in modelos.items():
        r = cross_validate(mod, X, y, cv=cv_report, scoring=scoring, n_jobs=-1)
        f1_folds[nombre] = r["test_F1_macro"]          # per-fold F1 (for Wilcoxon)
        fila = {"Model": nombre}
        for m in scoring:
            fila[m] = f"{r['test_'+m].mean():.3f} +/- {r['test_'+m].std():.3f}"
        filas.append(fila)
    tabla = pd.DataFrame(filas).set_index("Model").sort_values("F1_macro", ascending=False)
    return tabla, f1_folds

Consistency check of the combined matrix before running the cross-validation.

In [ ]:
X = np.vstack([Xtr, Xte])
y = np.concatenate([ytr, yte])

print("Xtr:", Xtr.shape, "| Xte:", Xte.shape)
print("Combined X:", X.shape, "| y:", y.shape)
print("Row counts consistent?", X.shape[0] == Xtr.shape[0] + Xte.shape[0] == len(y))
print("Class distribution:", np.bincount(y))

The cross-validation is repeated for each distance, and the winning (model, distance) pair under this more stable criterion is reported. It need not coincide with the winner of the single test-set evaluation in Section 2.3 — and where it does not, the cross-validated result is the one to trust.

In [ ]:
resultados_cv, f1_folds_cv = {}, {}

for metrica in METRICAS:
    Xtr, Xte, ytr, yte, _ = cargar_metrica(metrica)
    modelos = modelos_metrica[metrica]        # already tuned, with their best hyperparameters

    tabla_cv, f1_folds = reporte_cv(modelos, Xtr, Xte, ytr, yte)
    resultados_cv[metrica] = tabla_cv
    f1_folds_cv[metrica]   = f1_folds

    print(f"\n--- {metrica.upper()} · repeated CV ---")
    print(tabla_cv.to_string())

# ── Comparison of the four distances under repeated CV ───────────────────────
comparativa_cv = pd.concat({m: t["F1_macro"] for m, t in resultados_cv.items()}, axis=1)
print("\n=== F1-macro (mean +/- SD, as text) by model and distance — repeated CV ===")
print(comparativa_cv.to_string())

# The table holds strings ("0.654 +/- 0.048"); the mean is parsed back out to rank them
medias_cv = comparativa_cv.map(lambda s: float(s.split(" +/- ")[0]))
mejor_metrica_cv = medias_cv.max().idxmax()
mejor_modelo_cv  = medias_cv[mejor_metrica_cv].idxmax()
print(f"\nWinning distance (repeated CV): {mejor_metrica_cv}")
print(f"Winning model: {mejor_modelo_cv}  |  F1-macro: {medias_cv.loc[mejor_modelo_cv, mejor_metrica_cv]:.3f}")

# ── Wilcoxon between the two best models of the winning distance ─────────────
top2 = resultados_cv[mejor_metrica_cv].index[:2].tolist()
stat, p = wilcoxon(f1_folds_cv[mejor_metrica_cv][top2[0]], f1_folds_cv[mejor_metrica_cv][top2[1]])
print(f"\nWilcoxon {top2[0]} vs {top2[1]} ({mejor_metrica_cv}): p={p:.3f} -> "
      f"{'significant difference' if p < 0.05 else 'equivalent performance'}")

## 4. External validation — INC cohort (n = 23, Luminal B)

The models tuned on TCGA-BRCA under the winning distance (Jensen–Shannon) are applied to the INC cohort without any retraining or re-selection. Since every INC case is Luminal B, the reported metric is the recall for that class.

The same fixed models are evaluated against INC mosaics built under all four distances. Jensen–Shannon is the primary result; the other three are a sensitivity analysis showing how much the mosaic-construction choice affects the transfer.

In [ ]:
# ── External validation — INC cohort (n=23, Luminal B) ───────────────────────
# Fixed models: the classifiers already tuned on TCGA-BRCA with the winning
# distance (Jensen-Shannon). Evaluated on INC mosaics built under the four
# distances, with no retraining and no re-selection. JS is the primary result;
# EU/CN/BH are a sensitivity analysis.

RUTAS_INC = {
    "jensenshannon": "INC_mosaics/jensenshannon/KimiaNet_features.csv",
    "euclidean":     "INC_mosaics/euclidean/KimiaNet_features.csv",
    "cosine":        "INC_mosaics/cosine/KimiaNet_features.csv",
    "bhattacharyya": "INC_mosaics/bhattacharyya/KimiaNet_features.csv",
}

modelos_js = modelos_metrica["jensenshannon"]   # fixed weights, tuned on TCGA-BRCA


def cargar_inc(ruta):
    df = pd.read_csv(ruta)
    X = np.nan_to_num(df[FEATURE_COLS].values.astype(float))
    y = np.full(len(df), MAP["LumB"])   # all Luminal B by design of the sub-cohort
    return X, y


def evaluar_inc_lumb(modelos, X, y):
    filas = []
    for nombre, mod in modelos.items():
        yp = mod.predict(X)
        acierto = (yp == y).mean()   # fraction correctly classified as Luminal B
        filas.append({"Model": nombre, "Recall_LumB": round(acierto, 3)})
    return pd.DataFrame(filas).set_index("Model").sort_values("Recall_LumB", ascending=False)


# ── The JS models evaluated on the four INC mosaic constructions ─────────────
resultados_inc = {}
for metrica, ruta in RUTAS_INC.items():
    X_inc, y_inc = cargar_inc(ruta)
    tabla = evaluar_inc_lumb(modelos_js, X_inc, y_inc)
    resultados_inc[metrica] = tabla
    etiqueta = "PRIMARY RESULT" if metrica == "jensenshannon" else "sensitivity analysis"
    print(f"\n--- INC (n={len(y_inc)}) · {metrica.upper()} mosaic · JS models · {etiqueta} ---")
    print(tabla.to_string())

# ── Summary table ────────────────────────────────────────────────────────────
comparativa_inc = pd.concat(
    {m: t["Recall_LumB"] for m, t in resultados_inc.items()}, axis=1
).sort_values("jensenshannon", ascending=False)
print("\n=== Summary — LumB recall (JS models) by INC mosaic distance ===")
print(comparativa_inc.to_string())

### 4.1 Diagnosis of the transfer

A recall of zero across every model is not a performance figure; it means the predictions never land on Luminal B at all. The check below prints the feature dimensionality, the shape of the INC matrix and the distribution of the predicted classes, which distinguishes a genuine classification failure from a mismatch in the input (wrong number of columns, wrong patient set) or a systematic collapse into one class.

In [ ]:
print(len(FEATURE_COLS), FEATURE_COLS[:3], FEATURE_COLS[-3:])
print(X_inc.shape)                     # rows = INC cases, columns must be exactly 1024
for nombre, mod in modelos_js.items():
    print(f"{nombre}:", pd.Series(mod.predict(X_inc)).map({0: 'LumA', 1: 'LumB', 2: 'Basal'}).value_counts().to_dict())

## 5. Multilayer perceptron on the KimiaNet features

A fully connected network is added as a sixth classifier, to test whether a non-linear model extracts more subtype signal from the mosaic embeddings than the classical models do. The architecture is 512-256-128 with batch normalisation, ReLU and dropout, trained with AdamW, a cosine learning-rate schedule and class-weighted cross-entropy.

In [ ]:
# Multilayer network as a sixth classifier. It is evaluated on the fixed split
# (not by repeated CV) because of its training cost; this is stated in the
# methods. Labels follow the rest of the pipeline: LumA=0, LumB=1, Basal=2.

class MLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=(512, 256, 128), n_classes=3, dropout=0.4):
        super().__init__()
        capas, prev = [], input_dim
        for h in hidden_dims:
            capas += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        capas.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*capas)

    def forward(self, x):
        return self.net(x)


def entrenar_mlp(Xtr, Xte, ytr, yte, epochs=150, lr=1e-3, batch=32, seed=42):
    """Train the MLP and return (best_f1, y_pred_test, history).

    ytr/yte are already encoded 0/1/2 (LumA/LumB/Basal); no LabelEncoder is used.
    NOTE: the epoch kept is the one with the highest F1 on the TEST set, so the
    reported value is optimistic (see the notes at the end of the notebook).
    """
    torch.manual_seed(seed)

    # Standardisation (fitted on train only, no leakage)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xte_s = sc.transform(Xte)

    X_tr = torch.FloatTensor(Xtr_s); y_tr = torch.LongTensor(ytr)
    X_te = torch.FloatTensor(Xte_s); y_te = torch.LongTensor(yte)
    dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch, shuffle=True)

    model = MLP(input_dim=Xtr.shape[1])
    # Class weights for the imbalance
    counts  = np.bincount(ytr, minlength=3)
    weights = torch.FloatTensor(len(counts) / (len(counts) * counts))
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1, best_state = 0.0, None
    history = {"train_loss": [], "val_f1": []}

    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            tot += loss.item()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            vp = model(X_te).argmax(1).numpy()
        f1 = f1_score(y_te.numpy(), vp, average="macro")
        history["train_loss"].append(tot / len(dl))
        history["val_f1"].append(f1)
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        y_pred = model(X_te).argmax(1).numpy()
    return best_f1, y_pred, history

The MLP is trained once per distance, so its results sit alongside the classical models in the same comparison.

In [ ]:
# ── Train the MLP on every distance ──────────────────────────────────────────
resultados_mlp = {}

for metrica_mlp in METRICAS:
    print(f"\n{'='*60}\nMLP — distance: {metrica_mlp}\n{'='*60}")
    Xtr, Xte, ytr, yte, _ = cargar_metrica(metrica_mlp)
    best_f1, y_pred_mlp, history = entrenar_mlp(Xtr, Xte, ytr, yte)

    resultados_mlp[metrica_mlp] = {
        "best_f1": best_f1,
        "y_pred": y_pred_mlp,
        "y_true": yte,
        "history": history,
    }

    print(f"MLP ({metrica_mlp}) — best test F1-macro: {best_f1:.3f}\n")
    print(classification_report(yte, y_pred_mlp, labels=[0, 1, 2], target_names=CLASES))

    # Training curves + confusion matrix (one figure per distance)
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history["train_loss"], color="steelblue")
    ax[0].set_title("Train loss"); ax[0].set_xlabel("Epoch")

    ax[1].plot(history["val_f1"], color="orange")
    ax[1].axhline(best_f1, color="red", ls="--", label=f"Best F1={best_f1:.3f}")
    ax[1].set_title("Test F1-macro"); ax[1].set_xlabel("Epoch"); ax[1].legend()

    cm = confusion_matrix(yte, y_pred_mlp, labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(
        ax=ax[2], cmap="Blues", values_format="d", colorbar=False)
    ax[2].set_title(f"MLP — {metrica_mlp}")
    plt.tight_layout(); plt.show()

## 6. Notes and limitations

1. **External validation returns zero recall.** Under every model and every mosaic distance, no INC case is assigned to Luminal B; the predictions collapse almost entirely into Basal-like. This is a transfer failure rather than a performance measurement, and it should be resolved before the section is reported. The diagnostic in Section 4.1 is the starting point: check that the INC feature table has the same 1024 columns in the same order, that its rows are the expected cases (the run recorded 41 rows in one execution and 23 in another), and that the mosaics were built with the same patch pipeline and the same normalisation as the TCGA ones. A systematic shift in the embedding scale between centres, scanners and staining protocols is the most likely cause.
2. **Selection criteria are not aligned.** In Section 2.3 the distance is chosen by its single best F1 across models while the model is chosen by its mean across distances. The two criteria answer different questions and can pick a pair that is optimal under neither; the cross-validated ranking in Section 3 is the more reliable basis.
3. **MLP model selection.** The epoch kept is the one with the highest F1 on the test set, so the reported F1 is optimistic. A validation split drawn from the training set would be needed to report it as a clean held-out figure.
4. **Repeated cross-validation.** It is run on the whole cohort with hyperparameters tuned on the training partition, which slightly favours every model. Folds also share training data, so the per-fold scores are not independent and the Wilcoxon *p*-value is approximate.
5. **Inner merge.** `cargar_metrica` joins on the patient identifier with `how="inner"`, so slides without transcriptomic data, or patients without a mosaic, are dropped without warning. Comparing the resulting sizes against the expected 386 / 166 catches this.
6. **Duplicated patients.** If a patient contributes two slides, the 12-character barcode appears twice and the same patient may fall in both the training and the test set. Checking for duplicated identifiers before the merge rules this out.